In [146]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from training_models.classification_models import ClassificationModels
from joblib import dump

In [147]:
def undersampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para submuestrear
    undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=seed)

    #Se aplica el submuestreo
    X_res, y_res= undersampler.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [148]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    SMOTE = RandomUnderSampler(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= SMOTE.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [149]:
def split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [150]:
def train(train_v, validation_v, iteration, repr_name, div, seed):
    #Separa datos de sus target de entrenamiento y validacion
    train_values = train_v.drop(columns="target").values
    train_response = train_v["target"].values

    validation_values = validation_v.drop(columns="target").values
    validation_response = validation_v["target"].values

    print(f"Training model Random Forest, iteration: {iteration}")
    #Se instancia el objeto
    clf_model = ClassificationModels(X_train=train_values, X_val=validation_values, y_train=train_response, y_val=validation_response)
    #Se entrena el respectivo algoritmo con k-fold
    clf_model.instance_random_forest()
    clf_model.process_model(kfold=True, k=5)

    #Se guarda el modelo
    dump(clf_model.model, f"../../models/RandomForest_{div}_{iteration}_{repr_name}_seed{seed}.joblib")

    return clf_model.performances

In [151]:
rename_map = {
    "f1_weighted": "F1-score",
    "recall_weighted": "Recall",
    "precision_weighted": "Precision",
    "accuracy": "Accuracy"
}

In [152]:
def metrics(perf, iteration, seed, sampling):
    #Se obtienen las metricas de entrenamiento y validacion en variables diferentes
    train_metrics = perf["training_metrics"]
    val_metrics = perf["validation_metrics"]
    #Se elimina la matriz de confusiones
    val_metrics.pop("Confusion Matrix", None)
    #Renombra metricas
    train_renamed = {rename_map.get(k, k): v for k, v in train_metrics.items()}
    #Se asignan los valores de las metricas a un diccionario
    row = {
        "iteration": iteration,
        "seed": seed,
        "sampling": sampling
    }
    for metric_name in rename_map.values():
        row[f"Train_{metric_name}"] = round(train_renamed[metric_name], 4)
        row[f"Val_{metric_name}"] = round(val_metrics[metric_name], 4)
    
    return row

In [153]:
def main_train(df_data, repr_name, unique_seeds):
    all_metrics = []
    for i, seed in enumerate(unique_seeds):
        df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over=split(df_data, seed)
        perf_base=train(df_train, df_val, i, repr_name, 'base', seed)
        all_metrics.append(metrics(perf_base, i, seed, 'base'))
        perf_under=train(df_train_under, df_val_under, i, repr_name, 'undersampling', seed)
        all_metrics.append(metrics(perf_under, i, seed, 'undersampling'))
        perf_over=train(df_train_over, df_val_over, i, repr_name, 'oversampling', seed)
        all_metrics.append(metrics(perf_over, i, seed, 'oversampling'))            

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(f"../../models/metrics_{repr_name}_RandomForest.csv", index=False)

In [154]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [155]:
folder = "../../data/numerical_rep/"
unique_seeds = np.random.choice(range(100), size=30, replace=False)
#unique_seeds = [94, 42, 98, 43, 90, 44, 99, 93, 66, 34, 72, 60, 6, 39, 26, 74, 17,8, 51, 96, 53, 13, 20, 33, 29, 65, 46, 82, 79, 89]

In [156]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_{repr_name}.csv"
seeds_used = unique_seeds
main_train(df_data, repr_name, seeds_used)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing ProtT5
Training model Random Forest, iteration: 0
Training model Random Forest, iteration: 0
Training model Random Forest, iteration: 0
Training model Random Forest, iteration: 1
Training model Random Forest, iteration: 1
Training model Random Forest, iteration: 1
Training model Random Forest, iteration: 2
Training model Random Forest, iteration: 2
Training model Random Forest, iteration: 2
Training model Random Forest, iteration: 3
Training model Random Forest, iteration: 3
Training model Random Forest, iteration: 3
Training model Random Forest, iteration: 4
Training model Random Forest, iteration: 4
Training model Random Forest, iteration: 4
Training model Random Forest, iteration: 5
Training model Random Forest, iteration: 5
Training model Random Forest, iteration: 5
Training model Random Forest, iteration: 6
Training model Random Forest, iteration: 6
Training model Random Forest, iteration: 6
Training model Random Forest, iteration: 7
Training model Random Forest, iterat